---
## Paso 3: Implementación del Modelo CNN con Flax NNX

Objetivos:
- Implementar la arquitectura CNN usando exclusivamente `Flax NNX`
- Definir función de pérdida y optimizador con `Optax`
- Compilar el paso de entrenamiento con `jax.jit`
- Ejecutar el ciclo completo de entrenamiento
- Graficar curvas de pérdida y exactitud


### 3.1 Importaciones

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
import time
from flax import nnx
from functools import partial

print(f"JAX     : {jax.__version__}  —  backend: {jax.default_backend()}")
print(f"Flax    : {nnx.__module__.split('.')[0]}")
print(f"Optax   : {optax.__version__}")


### 3.2 Arquitectura CNN — Flax NNX

La arquitectura implementada extiende el mínimo requerido por el PDF:

```
Entrada (64×64×3)
    │
    ├─ Conv(32, 3×3) → BatchNorm → ReLU → MaxPool(2×2)
    ├─ Conv(64, 3×3) → BatchNorm → ReLU → MaxPool(2×2)
    ├─ Conv(128, 3×3) → BatchNorm → ReLU → MaxPool(2×2)
    │
    ├─ Flatten
    ├─ Dense(256) → ReLU → Dropout(0.4)
    └─ Dense(12)  → Salida (logits)
```

**Justificación de decisiones:**
- **3 bloques conv** en lugar de 2: el dataset tiene 12 clases con alta variabilidad visual
- **BatchNorm** en cada bloque: estabiliza el entrenamiento y actúa como regularizador
- **Dropout(0.4)** antes de la capa final: reduce sobreajuste dado el tamaño moderado del dataset
- **Logits sin softmax** en la salida: `optax.softmax_cross_entropy_with_integer_labels` lo aplica internamente


In [ ]:
class CNN(nnx.Module):
    """
    Red neuronal convolucional para clasificación de imágenes.

    Arquitectura:
        3 bloques Conv → BatchNorm → ReLU → MaxPool
        Flatten → Dense(256) → Dropout → Dense(num_clases)

    Parámetros
    ----------
    num_clases : int   – número de categorías de salida
    filtros    : int   – filtros del primer bloque (se duplican en cada bloque)
    neuronas   : int   – neuronas de la capa densa oculta
    tasa_drop  : float – tasa de dropout (0.0 = desactivado)
    rngs       : nnx.Rngs – generadores de números aleatorios de JAX/Flax
    """

    def __init__(self, num_clases, filtros=32, neuronas=256,
                 tasa_drop=0.4, rngs=nnx.Rngs(0)):

        # ── Bloque 1: Conv(filtros) ──────────────────────────────────────
        self.conv1 = nnx.Conv(
            in_features=3, out_features=filtros,
            kernel_size=(3, 3), padding="SAME", rngs=rngs
        )
        self.bn1 = nnx.BatchNorm(num_features=filtros, rngs=rngs)

        # ── Bloque 2: Conv(filtros*2) ────────────────────────────────────
        self.conv2 = nnx.Conv(
            in_features=filtros, out_features=filtros * 2,
            kernel_size=(3, 3), padding="SAME", rngs=rngs
        )
        self.bn2 = nnx.BatchNorm(num_features=filtros * 2, rngs=rngs)

        # ── Bloque 3: Conv(filtros*4) ────────────────────────────────────
        self.conv3 = nnx.Conv(
            in_features=filtros * 2, out_features=filtros * 4,
            kernel_size=(3, 3), padding="SAME", rngs=rngs
        )
        self.bn3 = nnx.BatchNorm(num_features=filtros * 4, rngs=rngs)

        # ── Capas densas ─────────────────────────────────────────────────
        # Después de 3 MaxPool(2×2) sobre imagen 64×64:
        # 64 → 32 → 16 → 8  →  8 * 8 * (filtros*4) neuronas
        dim_flat = 8 * 8 * (filtros * 4)
        self.dense1  = nnx.Linear(dim_flat, neuronas, rngs=rngs)
        self.dropout = nnx.Dropout(rate=tasa_drop, rngs=rngs)
        self.dense2  = nnx.Linear(neuronas, num_clases, rngs=rngs)

    def __call__(self, x, entrenamiento=False):
        """
        Propagación hacia adelante.

        Parámetros
        ----------
        x             : array (batch, alto, ancho, canales)
        entrenamiento : bool – activa BatchNorm y Dropout en modo entrenamiento
        """
        # Bloque 1
        x = self.conv1(x)
        x = self.bn1(x, use_running_average=not entrenamiento)
        x = jax.nn.relu(x)
        x = jax.lax.reduce_window(
            x, -jnp.inf, jax.lax.max,
            window_dimensions=(1, 2, 2, 1),
            window_strides=(1, 2, 2, 1),
            padding="VALID"
        )

        # Bloque 2
        x = self.conv2(x)
        x = self.bn2(x, use_running_average=not entrenamiento)
        x = jax.nn.relu(x)
        x = jax.lax.reduce_window(
            x, -jnp.inf, jax.lax.max,
            window_dimensions=(1, 2, 2, 1),
            window_strides=(1, 2, 2, 1),
            padding="VALID"
        )

        # Bloque 3
        x = self.conv3(x)
        x = self.bn3(x, use_running_average=not entrenamiento)
        x = jax.nn.relu(x)
        x = jax.lax.reduce_window(
            x, -jnp.inf, jax.lax.max,
            window_dimensions=(1, 2, 2, 1),
            window_strides=(1, 2, 2, 1),
            padding="VALID"
        )

        # Clasificador
        x = x.reshape((x.shape[0], -1))           # Flatten
        x = self.dense1(x)
        x = jax.nn.relu(x)
        x = self.dropout(x, deterministic=not entrenamiento)
        x = self.dense2(x)                         # logits (sin softmax)
        return x


# Instanciar modelo con hiperparámetros base
modelo = CNN(
    num_clases=NUM_CLASES,
    filtros=32,
    neuronas=256,
    tasa_drop=0.4,
    rngs=nnx.Rngs(SEMILLA)
)

# Contar parámetros
grafo, estado = nnx.split(modelo)
total_params = sum(x.size for x in jax.tree_util.tree_leaves(estado))
print(f"Modelo CNN instanciado correctamente.")
print(f"Total de parámetros : {total_params:,}")
print()

# Prueba de forma con un lote ficticio
x_prueba = jnp.ones((4, IMG_ALTO, IMG_ANCHO, IMG_CANALES))
logits   = modelo(x_prueba, entrenamiento=False)
print(f"Prueba de forma:")
print(f"  Entrada  : {x_prueba.shape}")
print(f"  Salida   : {logits.shape}  (esperado: (4, {NUM_CLASES}))")


### 3.3 Función de pérdida y métricas

In [ ]:
def perdida_fn(modelo, imagenes, etiquetas):
    """
    Calcula la pérdida de entropía cruzada categórica.
    optax aplica softmax internamente — el modelo retorna logits.
    """
    logits = modelo(imagenes, entrenamiento=True)
    perdida = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=etiquetas
    ).mean()
    return perdida, logits


def calcular_exactitud(logits, etiquetas):
    """Calcula la exactitud (accuracy) dado logits y etiquetas reales."""
    predicciones = jnp.argmax(logits, axis=-1)
    return jnp.mean(predicciones == etiquetas)


print("Función de pérdida  : softmax_cross_entropy_with_integer_labels")
print("Métrica principal   : exactitud (accuracy)")


### 3.4 Optimizador y paso de entrenamiento compilado con `jax.jit`

In [ ]:
# ── Hiperparámetros base ─────────────────────────────────────────────────
LEARNING_RATE = 1e-3
EPOCAS        = 20

# Optimizador Adam con decaimiento coseno del learning rate
scheduler = optax.cosine_decay_schedule(
    init_value=LEARNING_RATE,
    decay_steps=EPOCAS * 85          # 85 lotes por época aprox.
)
optimizador = nnx.Optimizer(modelo, optax.adam(scheduler))


@nnx.jit                              # compilación JIT de JAX/Flax
def paso_entrenamiento(modelo, optimizador, imagenes, etiquetas):
    """
    Ejecuta un paso de entrenamiento completo:
    1. Forward pass con diferenciación automática
    2. Cálculo de gradientes
    3. Actualización de parámetros con el optimizador
    """
    grad_fn = nnx.value_and_grad(perdida_fn, has_aux=True)
    (perdida, logits), gradientes = grad_fn(modelo, imagenes, etiquetas)
    optimizador.update(gradientes)
    exactitud = calcular_exactitud(logits, etiquetas)
    return perdida, exactitud


@nnx.jit
def paso_evaluacion(modelo, imagenes, etiquetas):
    """Ejecuta un paso de evaluación sin actualizar parámetros."""
    logits    = modelo(imagenes, entrenamiento=False)
    perdida   = optax.softmax_cross_entropy_with_integer_labels(
        logits=logits, labels=etiquetas
    ).mean()
    exactitud = calcular_exactitud(logits, etiquetas)
    return perdida, exactitud


print("Optimizador  : Adam con cosine decay schedule")
print(f"  LR inicial : {LEARNING_RATE}")
print(f"  Épocas     : {EPOCAS}")
print("Compilación  : @nnx.jit  (paso entrenamiento + evaluación)")


### 3.5 Ciclo completo de entrenamiento

In [ ]:
def entrenar(modelo, optimizador, ds_train, ds_val, epocas,
             nombre_exp="base"):
    """
    Ciclo completo de entrenamiento con registro de métricas por época.

    Retorna un diccionario con el historial de entrenamiento.
    """
    historial = {
        "perdida_train": [], "exactitud_train": [],
        "perdida_val":   [], "exactitud_val":   [],
        "tiempo_epoca":  []
    }

    t_inicio_total = time.time()
    print(f"{'Época':>6} {'P.Train':>9} {'Acc.Train':>10} "
          f"{'P.Val':>8} {'Acc.Val':>9} {'Tiempo':>8}")
    print("-" * 58)

    for epoca in range(1, epocas + 1):
        t0 = time.time()

        # ── Entrenamiento ───────────────────────────────────────────────
        perdidas_t, exactitudes_t = [], []
        for imagenes_tf, etiq_tf in ds_train:
            imgs = jnp.array(imagenes_tf.numpy())
            etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
            p, a = paso_entrenamiento(modelo, optimizador, imgs, etiq)
            perdidas_t.append(float(p))
            exactitudes_t.append(float(a))

        # ── Validación ──────────────────────────────────────────────────
        perdidas_v, exactitudes_v = [], []
        for imagenes_tf, etiq_tf in ds_val:
            imgs = jnp.array(imagenes_tf.numpy())
            etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
            p, a = paso_evaluacion(modelo, imgs, etiq)
            perdidas_v.append(float(p))
            exactitudes_v.append(float(a))

        t_epoca = time.time() - t0

        # Promedios de la época
        pt = np.mean(perdidas_t);   at = np.mean(exactitudes_t)
        pv = np.mean(perdidas_v);   av = np.mean(exactitudes_v)

        historial["perdida_train"].append(pt)
        historial["exactitud_train"].append(at)
        historial["perdida_val"].append(pv)
        historial["exactitud_val"].append(av)
        historial["tiempo_epoca"].append(t_epoca)

        print(f"{epoca:>6d} {pt:>9.4f} {at*100:>9.2f}% "
              f"{pv:>8.4f} {av*100:>8.2f}% {t_epoca:>7.1f}s")

    t_total = time.time() - t_inicio_total
    historial["tiempo_total"]    = t_total
    historial["t_media_epoca"]   = np.mean(historial["tiempo_epoca"])
    historial["exactitud_final_train"] = historial["exactitud_train"][-1]
    historial["exactitud_final_val"]   = historial["exactitud_val"][-1]

    print("-" * 58)
    print(f"Tiempo total        : {t_total:.1f} s")
    print(f"Tiempo medio/época  : {historial['t_media_epoca']:.1f} s")
    print(f"Exactitud train     : {historial['exactitud_final_train']*100:.2f}%")
    print(f"Exactitud val       : {historial['exactitud_final_val']*100:.2f}%")

    return historial


# ── Ejecutar entrenamiento base ──────────────────────────────────────────
print(f"Iniciando entrenamiento — {EPOCAS} épocas")
print(f"Configuración: LR={LEARNING_RATE}, batch={BATCH_SIZE}, "
      f"filtros=32, neuronas=256\n")

historial_base = entrenar(modelo, optimizador, ds_train, ds_val, EPOCAS)


### 3.6 Evaluación final en el conjunto de prueba

In [ ]:
perdidas_test, exactitudes_test = [], []
for imagenes_tf, etiq_tf in ds_test:
    imgs = jnp.array(imagenes_tf.numpy())
    etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
    p, a = paso_evaluacion(modelo, imgs, etiq)
    perdidas_test.append(float(p))
    exactitudes_test.append(float(a))

perdida_test   = np.mean(perdidas_test)
exactitud_test = np.mean(exactitudes_test)

print("=" * 45)
print("  Evaluación final — Conjunto de Prueba")
print("=" * 45)
print(f"  Pérdida   : {perdida_test:.4f}")
print(f"  Exactitud : {exactitud_test*100:.2f}%")
print("=" * 45)


### 3.7 Curvas de pérdida y exactitud

In [ ]:
def graficar_historial(historial, titulo="Entrenamiento base"):
    epocas_eje = range(1, len(historial["perdida_train"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.patch.set_facecolor("white")

    # ── Pérdida ──────────────────────────────────────────────────────────
    ax1.plot(epocas_eje, historial["perdida_train"],
             "o-", color="#2563EB", lw=2, ms=4, label="Train")
    ax1.plot(epocas_eje, historial["perdida_val"],
             "s--", color="#DC2626", lw=2, ms=4, label="Validación")
    ax1.set_xlabel("Época", fontsize=11)
    ax1.set_ylabel("Pérdida (cross-entropy)", fontsize=11)
    ax1.set_title("Curva de Pérdida", fontsize=12, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_facecolor("#F8FAFC")

    # ── Exactitud ────────────────────────────────────────────────────────
    ax2.plot(epocas_eje, [v*100 for v in historial["exactitud_train"]],
             "o-", color="#2563EB", lw=2, ms=4, label="Train")
    ax2.plot(epocas_eje, [v*100 for v in historial["exactitud_val"]],
             "s--", color="#DC2626", lw=2, ms=4, label="Validación")
    ax2.set_xlabel("Época", fontsize=11)
    ax2.set_ylabel("Exactitud (%)", fontsize=11)
    ax2.set_title("Curva de Exactitud", fontsize=12, fontweight="bold")
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor("#F8FAFC")

    fig.suptitle(titulo, fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


graficar_historial(historial_base,
    titulo=f"Modelo Base — LR={LEARNING_RATE}, batch={BATCH_SIZE}, "
           f"filtros=32, neuronas=256")


### 3.8 Resumen del entrenamiento base

In [ ]:
throughput = len(rutas_train) / historial_base["t_media_epoca"]

print("=" * 52)
print("  Resumen — Entrenamiento Base")
print("=" * 52)
print(f"  Arquitectura      : CNN 3 bloques conv + 2 dense")
print(f"  Filtros           : 32 → 64 → 128")
print(f"  Neuronas dense    : 256")
print(f"  Dropout           : 0.4")
print(f"  Optimizador       : Adam + cosine decay")
print(f"  Learning rate     : {LEARNING_RATE}")
print(f"  Épocas            : {EPOCAS}")
print(f"  Batch size        : {BATCH_SIZE}")
print()
print(f"  Tiempo total      : {historial_base['tiempo_total']:.1f} s")
print(f"  Tiempo/época      : {historial_base['t_media_epoca']:.1f} s")
print(f"  Throughput        : {throughput:.0f} imágenes/s")
print()
print(f"  Exactitud train   : {historial_base['exactitud_final_train']*100:.2f}%")
print(f"  Exactitud val     : {historial_base['exactitud_final_val']*100:.2f}%")
print(f"  Exactitud prueba  : {exactitud_test*100:.2f}%")
print("=" * 52)
print()
print("  Paso 3 completado — listo para experimentos.")
